# 06 – Robustness checks

Checks whether the main conclusions depend on analysis choices:

1. **Many tests:** Holm–Bonferroni correction across all agent-vs-human slope differences (RQ1, RQ3) and all equal-slopes tests (RQ2, RQ3).
2. **Repository threshold:** RQ1 restricted to repositories with at least 100 sampled agent PRs (instead of 50).
3. **Dominant agent:** RQ1 without Claude Code, which contributes over half of the agent PRs.
4. **Discussion comments:** human discussion comments outside reviews (steering addressed to an agent excluded) as an additional outcome.

Run after notebooks 03–05 (uses their result tables).

In [1]:
import os, sys
from pathlib import Path
ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT / "src"))
DATA = Path(os.environ.get("ESS_DATA", ROOT / "data"))        # override only for testing
RESULTS = Path(os.environ.get("ESS_RESULTS", ROOT / "results"))
import pandas as pd
import analysis as an
TAB = RESULTS / "tables"
df = an.load(DATA / "processed/prs.parquet")

## 1. Holm–Bonferroni correction

Family: every *Difference (AI − Human)* slope from RQ1 and RQ3 and every equal-slopes Wald test from RQ2 and RQ3.

In [2]:
diffs = pd.concat([pd.read_csv(TAB / "rq1_elasticities.csv").assign(rq="RQ1"),
                   pd.read_csv(TAB / "rq3_rq1_elasticities.csv").assign(rq="RQ3")])
diffs = diffs[diffs.group == "Difference (AI - Human)"].assign(test="AI - human slope", estimate=lambda d: d.elasticity)
walds = pd.concat([pd.read_csv(TAB / "rq2_equal_slopes_test.csv").assign(rq="RQ2"),
                   pd.read_csv(TAB / "rq3_rq2_equal_slopes_test.csv").assign(rq="RQ3 (agents)")])
walds = walds.assign(test="equal agent slopes", estimate=walds.chi2)
fam = pd.concat([diffs, walds], ignore_index=True)[["rq", "test", "outcome", "size", "estimate", "p"]]
fam["p_holm"] = an.holm(fam.p)
fam["significant_after_holm"] = fam.p_holm < 0.05
fam.to_csv(TAB / "robustness_holm.csv", index=False)
print(f"{len(fam)} tests, {(fam.p < 0.05).sum()} with p < 0.05, {fam.significant_after_holm.sum()} after Holm")
fam.round(4)

32 tests, 21 with p < 0.05, 12 after Holm


,rq,test,outcome,size,estimate,p,p_holm,significant_after_holm
0,RQ1,AI - human slope,Reviews,Changed lines (log),0.0124,0.0224,0.3185,False
1,RQ1,AI - human slope,Reviews,Changed files (log),0.0325,0.0006,0.0128,True
2,RQ1,AI - human slope,Inline review comments,Changed lines (log),0.0063,0.4471,1.0000,False
3,RQ1,AI - human slope,Inline review comments,Changed files (log),0.0115,0.3835,1.0000,False
4,RQ1,AI - human slope,Distinct reviewers,Changed lines (log),0.0130,0.0001,0.0024,True
5,RQ1,AI - human slope,Distinct reviewers,Changed files (log),0.0335,0.0000,0.0000,True
6,RQ3,AI - human slope,Reviews (human reviewers),Changed lines (log),-0.0035,0.5709,1.0000,False
7,RQ3,AI - human slope,Reviews (human reviewers),Changed files (log),0.0203,0.0589,0.6480,False
8,RQ3,AI - human slope,Reviews (bot reviewers),Changed lines (log),0.0284,0.0005,0.0122,True
9,RQ3,AI - human slope,Reviews (bot reviewers),Changed files (log),0.0390,0.0051,0.0926,False


## 2.–4. Re-estimated RQ1 models

Same model as RQ1 (PPML, repository and month fixed effects, SEs clustered by repository), changed lines as size.

In [3]:
n_agent = df[df.ai == 1].groupby("repo").size()
variants = {
    "Main sample": df,
    "Repositories with >= 100 agent PRs": df[df.repo.isin(n_agent[n_agent >= 100].index)],
    "Without Claude Code PRs": df[df.agent.ne("Claude_Code") | df.agent.isna()],
}
outcomes = an.OUTCOMES | {"comments_human": "Human discussion comments"}
rows = []
for name, d in variants.items():
    outs = outcomes if name == "Main sample" else an.OUTCOMES
    t = an.rq1_table(d, outcomes=outs, sizes={"log_lines": an.SIZE_MEASURES["log_lines"]})
    rr = an.rate_ratios(d, outcomes=outs, points=(100,))[["outcome", "rate_ratio", "ci_low", "ci_high"]]
    t = t.pivot_table(index="outcome", columns="group", values="elasticity", sort=False)
    t = t.join(rr.set_index("outcome").add_prefix("rr100_"))
    t.insert(0, "repositories", d.repo.nunique())
    t.insert(0, "PRs", len(d))
    rows.append(t.reset_index().assign(variant=name))
rob = pd.concat(rows, ignore_index=True)
rob = rob[["variant", "PRs", "repositories", "outcome", "Human", "AI agent", "Difference (AI - Human)",
           "rr100_rate_ratio", "rr100_ci_low", "rr100_ci_high"]]
rob.to_csv(TAB / "robustness_rq1_variants.csv", index=False)
rob.round(3)

.venv/lib/python3.14/site-packages/pyfixest/estimation/formula/model_matrix.py:151: UserWarning: 1 singleton fixed effect(s) dropped from the model.
  warnings.warn(
.venv/lib/python3.14/site-packages/pyfixest/estimation/models/fepois_.py:576: UserWarning: 9557 observations removed because of separation.
  warnings.warn(


.venv/lib/python3.14/site-packages/pyfixest/estimation/formula/model_matrix.py:151: UserWarning: 1 singleton fixed effect(s) dropped from the model.
  warnings.warn(
.venv/lib/python3.14/site-packages/pyfixest/estimation/models/fepois_.py:576: UserWarning: 17588 observations removed because of separation.
  warnings.warn(


.venv/lib/python3.14/site-packages/pyfixest/estimation/formula/model_matrix.py:151: UserWarning: 1 singleton fixed effect(s) dropped from the model.
  warnings.warn(
.venv/lib/python3.14/site-packages/pyfixest/estimation/models/fepois_.py:576: UserWarning: 9557 observations removed because of separation.
  warnings.warn(


.venv/lib/python3.14/site-packages/pyfixest/estimation/formula/model_matrix.py:151: UserWarning: 1 singleton fixed effect(s) dropped from the model.
  warnings.warn(
.venv/lib/python3.14/site-packages/pyfixest/estimation/models/fepois_.py:576: UserWarning: 13744 observations removed because of separation.
  warnings.warn(


.venv/lib/python3.14/site-packages/pyfixest/estimation/formula/model_matrix.py:151: UserWarning: 1 singleton fixed effect(s) dropped from the model.
  warnings.warn(
.venv/lib/python3.14/site-packages/pyfixest/estimation/models/fepois_.py:576: UserWarning: 9557 observations removed because of separation.
  warnings.warn(


...

.venv/lib/python3.14/site-packages/pyfixest/estimation/formula/model_matrix.py:151: UserWarning: 7 singleton fixed effect(s) dropped from the model.
  warnings.warn(
.venv/lib/python3.14/site-packages/pyfixest/estimation/models/fepois_.py:576: UserWarning: 5390 observations removed because of separation.
  warnings.warn(


.venv/lib/python3.14/site-packages/pyfixest/estimation/formula/model_matrix.py:151: UserWarning: 7 singleton fixed effect(s) dropped from the model.
  warnings.warn(
.venv/lib/python3.14/site-packages/pyfixest/estimation/models/fepois_.py:576: UserWarning: 5390 observations removed because of separation.
  warnings.warn(


.venv/lib/python3.14/site-packages/pyfixest/estimation/formula/model_matrix.py:151: UserWarning: 7 singleton fixed effect(s) dropped from the model.
  warnings.warn(
.venv/lib/python3.14/site-packages/pyfixest/estimation/models/fepois_.py:576: UserWarning: 9319 observations removed because of separation.
  warnings.warn(


.venv/lib/python3.14/site-packages/pyfixest/estimation/formula/model_matrix.py:151: UserWarning: 7 singleton fixed effect(s) dropped from the model.
  warnings.warn(
.venv/lib/python3.14/site-packages/pyfixest/estimation/models/fepois_.py:576: UserWarning: 5390 observations removed because of separation.
  warnings.warn(


,variant,PRs,repositories,outcome,Human,AI agent,Difference (AI - Human),rr100_rate_ratio,rr100_ci_low,rr100_ci_high
0,Main sample,306512,1248,Reviews,0.165,0.178,0.012,0.874,0.845,0.903
1,Main sample,306512,1248,Inline review comments,0.348,0.354,0.006,0.829,0.792,0.867
2,Main sample,306512,1248,Distinct reviewers,0.048,0.061,0.013,0.851,0.830,0.873
3,Main sample,306512,1248,Human discussion comments,0.122,0.129,0.006,0.894,0.839,0.953
4,Repositories with >= 100 agent PRs,213487,593,Reviews,0.169,0.181,0.012,0.877,0.840,0.916
5,Repositories with >= 100 agent PRs,213487,593,Inline review comments,0.350,0.356,0.007,0.828,0.781,0.877
6,Repositories with >= 100 agent PRs,213487,593,Distinct reviewers,0.050,0.063,0.013,0.856,0.829,0.883
7,Without Claude Code PRs,225253,1242,Reviews,0.167,0.169,0.002,0.847,0.802,0.894
8,Without Claude Code PRs,225253,1242,Inline review comments,0.350,0.342,-0.008,0.745,0.695,0.800
9,Without Claude Code PRs,225253,1242,Distinct reviewers,0.048,0.069,0.021,0.795,0.764,0.827


In [4]:
# p-values of the slope difference in each variant
pd.concat([an.rq1_table(d, sizes={"log_lines": an.SIZE_MEASURES["log_lines"]}).assign(variant=name)
           for name, d in list(variants.items())[1:]]).query("group == 'Difference (AI - Human)'")[
    ["variant", "outcome", "elasticity", "p"]].round(4)

.venv/lib/python3.14/site-packages/pyfixest/estimation/models/fepois_.py:576: UserWarning: 6516 observations removed because of separation.
  warnings.warn(


.venv/lib/python3.14/site-packages/pyfixest/estimation/models/fepois_.py:576: UserWarning: 11397 observations removed because of separation.
  warnings.warn(


.venv/lib/python3.14/site-packages/pyfixest/estimation/models/fepois_.py:576: UserWarning: 6516 observations removed because of separation.
  warnings.warn(


.venv/lib/python3.14/site-packages/pyfixest/estimation/formula/model_matrix.py:151: UserWarning: 7 singleton fixed effect(s) dropped from the model.
  warnings.warn(
.venv/lib/python3.14/site-packages/pyfixest/estimation/models/fepois_.py:576: UserWarning: 5390 observations removed because of separation.
  warnings.warn(


.venv/lib/python3.14/site-packages/pyfixest/estimation/formula/model_matrix.py:151: UserWarning: 7 singleton fixed effect(s) dropped from the model.
  warnings.warn(
.venv/lib/python3.14/site-packages/pyfixest/estimation/models/fepois_.py:576: UserWarning: 9319 observations removed because of separation.
  warnings.warn(


.venv/lib/python3.14/site-packages/pyfixest/estimation/formula/model_matrix.py:151: UserWarning: 7 singleton fixed effect(s) dropped from the model.
  warnings.warn(
.venv/lib/python3.14/site-packages/pyfixest/estimation/models/fepois_.py:576: UserWarning: 5390 observations removed because of separation.
  warnings.warn(


,variant,outcome,elasticity,p
2,Repositories with >= 100 agent PRs,Reviews,0.0118,0.0773
5,Repositories with >= 100 agent PRs,Inline review comments,0.0066,0.5318
8,Repositories with >= 100 agent PRs,Distinct reviewers,0.0129,0.0023
2,Without Claude Code PRs,Reviews,0.0021,0.7569
5,Without Claude Code PRs,Inline review comments,-0.0082,0.4584
8,Without Claude Code PRs,Distinct reviewers,0.0210,0.0000


## 5. How to read the results

* Conclusions should rest on tests that remain significant after the Holm correction.
* If elasticities and rate ratios in the restricted samples stay close to the main sample, the results do not depend on the 50-PR threshold or on Claude Code.
* Discussion comments are not part of review activity in the main analysis; a similar pattern there shows the result does not hinge on that definition.